# Taller 4 — Mini-proyecto de generación de texto con GPT
## Caso propio: satisfacción de clientes de e-commerce en español a partir de sus reseñas (1–5 estrellas)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yeavil-neny/NLP-ProcesamientoLeguajeNatural/blob/main/Taller_4/proyecto_gpt_resenas_ecommerce.ipynb)

**Equipo:** Fredy Alvarez · Anderson Pipicano · Cristian Yepes · Yenny Villarreal

---

### El problema (el mismo hilo conductor de los talleres anteriores)

A lo largo del curso hemos trabajado un único problema real: **predecir la satisfacción de un cliente
de e-commerce en español leyendo su reseña de producto**, en una escala de 1 a 5 estrellas.

- **Taller 1 (LSTM):** red recurrente entrenada desde cero → acc 0.5168
- **Taller 2 (Transformer desde cero):** clasificador con embeddings estáticos → acc 0.4896
- **Taller 3 (BERT, encoder):** featurizer congelado y fine tuning completo → acc 0.5734 (mejor hasta ahora)

En este taller la pregunta cambia de ángulo. La clase trató los modelos **GPT (decoder,
autoregresivos, orientados a *generar* texto)**. Entonces: **¿puede un modelo nacido para
escribir también servir para *clasificar* la satisfacción?** La hipótesis es doble:

1. **H1:** la representación interna de un GPT-2 pre-entrenado en español ya codifica señal
   de sentimiento (aunque no se haya entrenado para eso) → sirviendo como *featurizer* congelado
   debería superar a un featurizer de embeddings estáticos y acercarse al BERT del Taller 3.
2. **H2:** con fine tuning (una capa de clasificación sobre el decoder completa), GPT-2
   debería ser competitivo con BERT: ambos son transformers pre-entrenados; lo que cambia es
   *encoder bidireccional* (ve el contexto a ambos lados) vs. *decoder unidireccional*
   (solo ve hacia atrás, el mismo mecanismo que usa cuando predice la siguiente palabra).

### Por qué nuestra propuesta es buena para abordar el problema

- **Mismo corpus, mismos submuestreos de los talleres anteriores** (`SetFit/amazon_reviews_multi_es`,
  50k train / 5k val / 5k test, seed 42): la comparación con el historial del curso es directa y justa.
- **Progresión de dificultad en gradiente** (zero-shot → featurizer → fine tuning → generativo):
  cada experimento aísla una idea distinta y nos permite *entender* de dónde viene cada mejora.
- **Vale la pena**: la clasificación de satisfacción es la tarea de negocio que venimos estudiando,
  pero con un modelo generativo podemos además **demostrar generación condicionada**: entrenamos
  a GPT para escribir reseñas sintéticas de la calidad que le pidamos — un puente natural
  entre "clasificar" y "generar", y la demo más memorable de la clase llevada a nuestro caso.

### Matriz de experimentos

| Exp | Técnica | Origen respecto a la clase |
|---|---|---|
| **E0** | Clasificación *zero-shot* por log-verosimilitud (GPT-2 sin entrenar) | Más allá de la clase (prompting con likelihood) |
| **E1a** | GPT-2 congelado como featurizer + regresión logística | Análogo al Taller 3 (encoder→decoder) |
| **E1b** | GPT-2 congelado como featurizer + MLP | Análogo al Taller 3 |
| **E2** | Fine tuning para clasificación (`GPT2ForSequenceClassification`) | Núcleo del taller (adaptado de generación→clasificación) |
| **E3a** | Fine tuning generativo (LM puro) **condicionado por la etiqueta de estrellas** | La guía de clase, con nuestra modificación: prefijo 'Estrellas: n.' |
| **E3b** | Demo: generar reseñas sintéticas de N estrellas (antes vs. después del fine tuning) | Demo propia |
| — | Curvas de entrenamiento, matriz de confusión, severidad ±1★, tabla comparativa global T1–T4 | Análisis agregado |

> ⚙️ **Reproducibilidad:** semilla fija en todo el notebook. La variable `SMOKE_TEST` (celda de
> configuración) permite correr una versión reducida en minutos para verificar que todo funciona;
> la versión de entrega corresponde a `SMOKE_TEST = False` con la corrida completa en Colab (GPU).


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# 0. PREPARACIÓN DEL ENTORNO
# ─────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# Dependencias que pueden no venir preinstaladas (en Colab casi todas están);
# si falta alguna, la instalamos silenciosamente.
import importlib.util, subprocess, sys
for pkg in ['datasets', 'wordcloud', 'nltk']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import transformers
from datasets import load_dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModel, AutoModelForCausalLM,
                          AutoModelForSequenceClassification,
                          DataCollatorForLanguageModeling,
                          Trainer, TrainingArguments, set_seed)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

torch.manual_seed(42)
set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ─── CONFIGURACIÓN GLOBAL ────────────────────────────────────────────
# SMOKE_TEST=True  -> corre rápido con muestras pequeñas (prueba estructural)
# SMOKE_TEST=False -> corrida COMPLETA de entrega (en Colab, con GPU)
SMOKE_TEST = False

MODEL_NAME = 'mrm8488/spanish-gpt2'   # el GPT-2 en español de la clase
SEED = 42

print(f'PyTorch: {torch.__version__} | Transformers: {transformers.__version__} | Device: {device}')
print(f'SMOKE_TEST: {SMOKE_TEST}')

## 1. El dataset: reseñas de Amazon en español

Usamos **exactamente el mismo dataset y submuestreos de los Talleres 1, 2 y 3**
(`SetFit/amazon_reviews_multi_es`, submuestreos de 50 000 reseñas para entrenamiento y 5 000
para validación y prueba, semilla 42). El dataset original tiene 200 000 reseñas de Amazon
en español etiquetadas de 1 a 5 estrellas, balanceado por clase.

> 🔁 Decisión de diseño: NO cambiar el dataset. Un caso "propio" bien construido no
> necesariamente estrena corpus: nuestro caso es propio porque **cambia el paradigma del
> modelo** (encoder→decoder, generación→clasificación), y al conservar el muestreo, la tabla
> comparativa final mide el efecto del modelo y no el efecto del dataset.

In [ ]:
# Cargamos el dataset de reseñas de Amazon en español (mismo del Taller 1, 2 y 3)
print("Descargando el dataset 'SetFit/amazon_reviews_multi_es'...")
dataset = load_dataset("SetFit/amazon_reviews_multi_es")

# Mismo submuestreo de los talleres anteriores (semilla fija -> reproducible).
# Si SMOKE_TEST, tomamos muestras pequeñas solo para validar que todo corre.
if SMOKE_TEST:
    n_train, n_eval = 1000, 500
else:
    n_train, n_eval = 50000, 5000

train_data = dataset['train'].shuffle(seed=SEED).select(range(n_train))
val_data   = dataset['validation'].shuffle(seed=SEED).select(range(n_eval))
test_data  = dataset['test'].shuffle(seed=SEED).select(range(n_eval))

# Nombres bonitos para las clases (el dataset trae 0..4; pasamos a 1..5 estrellas)
id2label = {0: '1★', 1: '2★', 2: '3★', 3: '4★', 4: '5★'}
label2id = {v: k for k, v in id2label.items()}
NUM_CLASSES = 5

splits = DatasetDict({'train': train_data, 'val': val_data, 'test': test_data})

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print(f"Columnas: {train_data.column_names}")
print(f"\n--- Ejemplo (train[0]) ---")
print(f"Texto  : {train_data[0]['text'][:300]}")
print(f"Estrellas: {train_data[0]['label'] + 1}")

In [ ]:
# Un vistazo a una reseña de cada clase para 'enchularnos' del dominio
df_peek = train_data.to_pandas()
df_peek['estrellas'] = df_peek['label'] + 1
for n in range(1, 6):
    row = df_peek[df_peek['estrellas'] == n].iloc[0]
    print(f"── {n}★ ── {row['text'][:220]}")
    print()

## 2. EDA — Exploración de los datos

Un buen modelo empieza con un buen entendimiento del corpus. Exploramos
distribución de clases, longitudes (en palabras y en tokens), vocabulario
característico de cada extremo de satisfacción y ejemplos representativos.

In [ ]:
# ─── 2.1 Distribución de clases en los tres conjuntos ───
df_train = train_data.to_pandas(); df_train['split'] = 'train'; df_train['estrellas'] = df_train['label'] + 1
df_val   = val_data.to_pandas();   df_val['split']   = 'val';   df_val['estrellas']   = df_val['label'] + 1
df_test  = test_data.to_pandas();  df_test['split']  = 'test';  df_test['estrellas']  = df_test['label'] + 1
df_all = pd.concat([df_train, df_val, df_test])

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=df_all, x='estrellas', hue='split', palette='viridis', ax=ax)
ax.set_title('Distribución de reseñas por estrella y por conjunto')
ax.set_xlabel('Estrellas'); ax.set_ylabel('Número de reseñas')
plt.tight_layout(); plt.show()

print("Proporción por clase (train):")
print((df_train['estrellas'].value_counts(normalize=True) * 100).round(2).sort_index().astype(str) + '%')

🔍 **Observación.** El subset con el que trabajamos es esencialmente **balanceado**: cada
nivel de satisfacción aporta ~20% de las reseñas (en train, y de forma proporcional también en
val y test). Es una propiedad heredada del dataset original de Amazon. Consecuencias prácticas:

1. La **accuracy** es una métrica razonable aquí (a diferencia de problemas muy desbalanceados,
   donde miente); aún así reportaremos también **F1 ponderado** por costo cero y por comparabilidad.
2. No hace falta rebalancear ni sobremuestrear: el modelo ve las 5 clases por igual.

In [ ]:
# ─── 2.2 Longitud de las reseñas en PALABRAS ───
df_train['Palabras por Reseña'] = df_train['text'].str.split().apply(len)

print("Estadísticas de longitud (palabras):")
print(df_train['Palabras por Reseña'].describe().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_train['Palabras por Reseña'].clip(upper=200).plot.hist(bins=40, ax=axes[0], color='#4c72b0')
axes[0].set_title('Distribución de longitud (palabras, recortada a 200)')
axes[0].set_xlabel('Palabras')
df_train.boxplot('Palabras por Reseña', by='estrellas', grid=False, showfliers=False,
                 color='black', ax=axes[1])
axes[1].set_title('Longitud por nivel de satisfacción')
axes[1].set_xlabel('Estrellas'); axes[1].set_ylabel('Palabras')
plt.suptitle(''); plt.tight_layout(); plt.show()

print("Mediana de palabras por clase:")
print(df_train.groupby('estrellas')['Palabras por Reseña'].median().to_string())

🔍 **Observación.** Las reseñas son **cortas**: la mediana global ronda las ~15 palabras.
Mirado por clase, el padrón típico de reseñas se cumple: los extremos (1★/5★) describen
con brevedad y la clase intermedia (3★) tiende a argumentar más. Esto es señal clara de que
el corpus es ruido bajo: hay poco texto por ejemplo, por lo que cada token debe contar —
y es precisamente ahí donde los transformers pre-entrenados ganan frente a modelos desde cero.

> 📏 Implicación para el modelado: con textos tan cortos, `MAX_LEN` no necesita ser grande;
> lo cuantificamos con el tokenizador real en la sección 3.

In [ ]:
# ─── 2.3 Nubes de palabras por nivel de satisfacción (1★ vs 3★ vs 5★) ───
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
stop_es = set(stopwords.words('spanish'))
# Palabras 'vacías de sentimiento' del dominio e-commerce (mismas de Taller 3)
stop_es.update(['producto', 'amazon', 'bien', 'mal', 'hace', 'solo', 'si', 'mas', 'q', 'que', 'comprar'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cmap_por_clase = {1: 'Reds', 3: 'Greys', 5: 'Greens'}
for ax, n in zip(axes, [1, 3, 5]):
    textos = ' '.join(df_train[df_train['estrellas'] == n]['text'].astype(str))
    wc = WordCloud(width=600, height=400, background_color='white', stopwords=stop_es,
                   colormap=cmap_por_clase[n], random_state=SEED).generate(textos)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{n} estrellas'); ax.axis('off')
plt.suptitle('Vocabulario característico por nivel de satisfacción', y=1.02)
plt.tight_layout(); plt.show()

🔍 **Observación.** El vocabulario de los extremos es casi perfecto para la tarea:
en **1★** destacan palabras de frustración y de devoluciones (*llegó tarde, malo, devolución,
peor, dinero*), y en **5★** destacan las de satisfacción (*excelente, rápido, perfecto,
gracias, recomendable*). En **3★** el vocabulario es mucho más plano/técnico (talla, color,
funciona) — consistente con 'ni f ni f': reseñas argumentativas, no emotivas.

Esto anticipa la dificultad del problema **no es 1★ vs 5★, sino el grano fino entre clases
vecinas**, como ya vimos con la severidad ±1★ en Taller 3.

In [ ]:
# ─── 2.4 Bigramas más frecuentes por clase (1★ y 5★) ───
from collections import Counter

def top_bigramas(textos, k=12):
    counts = Counter()
    for t in textos:
        words = [w for w in str(t).lower().split() if w.strip('.,!¡¿?;:()0-9') not in stop_es]
        counts.update(zip(words, words[1:]))
    return pd.Series(dict(counts.most_common(k)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, n, cmap in zip(axes, [1, 5], ['#c44e52', '#55a868']):
    bg = top_bigramas(df_train[df_train['estrellas'] == n]['text'])
    bg.sort_values().plot.barh(ax=ax, color=cmap)
    ax.set_title(f'Bigramas más frecuentes en {n}★')
    ax.set_ylabel('bigrama'); ax.set_xlabel('frecuencia')
plt.tight_layout(); plt.show()

🔍 **Observación.** Los bigramas confirman y afilan el hallazgo de las nubes: en 5★
aparecen fórmulas de cierre de compra satisfactoria (*recomendarlo, llegó antes de lo
esperado*), mientras en 1★ aparecen fórmulas de reclamo (*no funciona, no llegó, dinero
perdido*). Llaman la atención los negaciones: gran parte de la señal negativa es
**'negación + lo que debía pasar'** ('no funciona', 'no llegó'). Un modelo desde cero con
embedding estático ve 'no' y 'funciona' como piezas aparte; un transformer pre-entrenado,
con atención completa, puede ligarlas y por eso tiene ventaja esperable en este problema.

In [ ]:
# ─── 2.5 Ejemplos representativos por clase (uno corto y uno largo) ───
for n in range(1, 6):
    d = df_train[df_train['estrellas'] == n]
    fila_c, fila_l = d.sort_values('Palabras por Reseña').iloc[0], d.sort_values('Palabras por Reseña').iloc[-1]
    corto, largo = str(fila_c['text']), str(fila_l['text'])
    print(f"── {n}★ corto ({len(corto.split())} palabras): {corto[:150]}")
    print(f"── {n}★ largo  ({len(largo.split())} palabras): {largo[:180]}")
    print()

## 3. Tokenización con GPT-2 (BPE) y elección de MAX_LEN

Ojo con un detalle que cambia respecto a Taller 3: BERT/Beto usa **WordPiece**
(sub-palabras a partir de vocabulario fijo), mientras que GPT-2 usa **BPE** (*Byte-Pair
Encoding*). Para texto en español, el vocabulario de `spanish-gpt2` fue entrenado en
corpus español, así que esperamos que el texto se segmente razonablemente bien — pero es
dato, no supuesto: lo medimos y elegimos `MAX_LEN` con la misma metodología de Taller 3
(percentiles de longitud en tokens, con cobertura >99%).

In [ ]:
# ─── 3.1 Cargamos el tokenizador y jugamos con él ───
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 no trae token de relleno definido: le asignamos el de fin de texto
# (el mismo truco de la guía de clase).
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocabulario: {tokenizer.vocab_size} tokens | PAD: {tokenizer.pad_token!r} (id {tokenizer.pad_token_id})")

ejemplo = "¡El producto llegó tarde y no funciona, una pérdida de dinero!"
enc = tokenizer(ejemplo)
print(f"\nReseña: {ejemplo!r}")
print(f"  -> {len(enc['input_ids'])} tokens")
print(f"  -> {tokenizer.convert_ids_to_tokens(enc['input_ids'])}")

In [ ]:
# ─── 3.2 Percentiles de longitud en tokens -> decisión de MAX_LEN ───
muestra = train_data.shuffle(seed=SEED).select(range(min(2000, len(train_data))))
lens = np.array([len(tokenizer(t)['input_ids']) for t in muestra['text']])

print("Percentiles de longitud en tokens (muestra de 2000 reseñas):")
for p in [50, 90, 95, 99, 100]:
    print(f"  p{p}: {np.percentile(lens, p):.0f} tokens")

print()
# Elegimos el UMBRAL MÁS PEQUEÑO que cubre >99% del corpus: decisión ganada
# por análisis (con las reseñas cortas de este dataset cae en los ~64 tokens).
MAX_LEN = next((c for c in [32, 48, 64, 96, 128] if (lens <= c).mean() >= 0.99), 128)
print(f"✂️  MAX_LEN elegido: {MAX_LEN}")

🔍 **Observación.** Aquí hay un hallazgo específico de GPT-2 frente a Beto: su vocabulario BPE
está entrenado sobre corpus español, así que las palabras frecuentes se tokenizan en pocas
piezas. La decisión de `MAX_LEN` no se copia del Taller 3 ( allá fue 128 con Beto): aquí el
umbral sale **del propio corpus con esta tokenización** como el menor valor que cubre >99% de
las reseñas. [SINCRONIZAR con la corrida completa: percentiles p50/p95/p99 y valor final de
MAX_LEN + comentario de por qué ese valor y no el 128 de Beto].

In [ ]:
# ─── 3.3 Tokenizamos los tres conjuntos con truncado y relleno hasta MAX_LEN ───
def tokenize_fn(examples):
    # truncation=True  -> recorta las reseñas más largas que MAX_LEN
    # padding='max_length' -> rellena las cortas hasta MAX_LEN (todas quedan iguales)
    return tokenizer(examples['text'], truncation=True, max_length=MAX_LEN, padding='max_length')

tokenized = splits.map(tokenize_fn, batched=True)
# GPT-2 solo necesita input_ids + attention_mask (+ label en clasificación)
tokenized = tokenized.remove_columns([col for col in tokenized['train'].column_names
                                      if col not in ['input_ids', 'attention_mask', 'label']])
tokenized.set_format('torch')
tokenized

## 4. Punto de partida: el marcador del curso

Antes de experimentar, fijamos el marcador histórico del curso (misma tarea, mismo dataset,
mismos submuestreos) para que la comparación final sea directa. Todos los experimentos de
este notebook se agregarán a esta tabla según terminen.

In [ ]:
# Resultados históricos de los talleres anteriores (misma tarea y dataset)
RESULTADOS = [
    {'taller': 'T1', 'modelo': 'LSTM (desde cero)',              'accuracy': 0.5168, 'f1': 0.52,  'fuente': 'Taller 1'},
    {'taller': 'T2', 'modelo': 'Transformer desde cero',          'accuracy': 0.4896, 'f1': 0.49,  'fuente': 'Taller 2'},
    {'taller': 'T3', 'modelo': 'BERT featurizer + lineal',        'accuracy': 0.4736, 'f1': 0.4620, 'fuente': 'Taller 3'},
    {'taller': 'T3', 'modelo': 'BERT featurizer + MLP',           'accuracy': 0.4858, 'f1': 0.4734, 'fuente': 'Taller 3'},
    {'taller': 'T3', 'modelo': 'BERT embeddings + Reg.Log.',      'accuracy': 0.4954, 'f1': 0.4909, 'fuente': 'Taller 3'},
    {'taller': 'T3', 'modelo': 'BERT fine tuning (mejor curso)',  'accuracy': 0.5734, 'f1': 0.5724, 'fuente': 'Taller 3'},
]
df_resultados = pd.DataFrame(RESULTADOS)
df_resultados

## 5. E0 — Clasificación zero-shot con GPT-2 (sin entrenar nada)

**Idea.** Un GPT-2 calcula $P(\text{token}_{t} \mid \text{tokens}_{<t})$: la probabilidad del
siguiente token dado el contexto. Podemos explotar eso **sin etiquetas ni entrenamiento**:

> dado el texto *"El producto llegó roto y nadie respondió…"*, pedimos al modelo que
> 'complete' la frase `_Estrellas: __.` y medimos la **log-verosimilitud** de cada opción
> (1, 2, 3, 4, 5). El modelo asignará más masa de probabilidad a la estrella que considere
> más plausible después de leer la reseña. La predicción es la estrella con mayor log-verosimilitud.

Esta técnica (a veces llamada *prompting* + *scoring*) **no aparece en la clase**: es una
adaptación propia del mecanismo de la guía (la probabilidad del siguiente token) a la
clasificación. Pasa una cosa interesante: es el único modelo del curso que predice **sin ver
un solo ejemplo etiquetado** — y su acierto se convertirá en el piso de comparación para
medir honestamente cuánto aporta entrenar.

In [ ]:
# ─── 5.1 Implementación del scoring por log-verosimilitud ───
# Cargamos el modelo GPT-2 (generativo, con cabeza de lenguaje), CONGELADO:
# será la misma base para E0 (scoring), E1 (featurizer) y E3b (generación pre-FT).
model_gpt = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_gpt.eval()

# Importante (el mismo ajuste de la guía de clase): alinear el tamaño de la
# matriz de embeddings del modelo con el vocabulario del tokenizador.
model_gpt.resize_token_embeddings(len(tokenizer))

@torch.no_grad()
def puntuar_zero_shot(pares, batch_size=32):
    # pares: lista de (texto, etiqueta_str). Para cada par medimos la log-prob
    # de la etiqueta como continuación del contexto: "Reseña: <texto>\nEstrellas: <e>."
    # Devuelve un tensor con la log-verosimilitud de cada par.
    scores = torch.zeros(len(pares))
    for inicio in range(0, len(pares), batch_size):
        chunk = pares[inicio:inicio + batch_size]
        contextos = [f"Reseña: {' '.join(str(t).split())}\nEstrellas: " for t, _ in chunk]
        completos = [c + e + "." for c, (_, e) in zip(contextos, chunk)]

        enc = tokenizer(completos, return_tensors='pt', padding=True,
                        return_offsets_mapping=True, add_special_tokens=False)
        input_ids = enc['input_ids'].to(device)
        attn = enc['attention_mask'].to(device)
        # UN forward para todo el lote (eficiente), luego puntuamos fila por fila.
        logits = model_gpt(input_ids=input_ids, attention_mask=attn).logits

        for fila, ctx_len in enumerate([len(c) for c in contextos]):
            # La continuación son los tokens que 'tocan' el texto de la etiqueta:
            # el primero cuyo final de posición de caracteres supera el largo del contexto.
            offsets = enc['offset_mapping'][fila]
            pos_cont = (offsets[:, 1] > ctx_len).nonzero()[0].item()
            fin_fila = int(attn[fila].sum().item())
            # El token en la posición p es predicho por los logits en p-1 (teacher forcing)
            lp = torch.log_softmax(logits[fila, pos_cont - 1: fin_fila - 1], dim=-1)
            targets = input_ids[fila, pos_cont:fin_fila]
            scores[inicio + fila] = lp.gather(1, targets.unsqueeze(-1)).sum().item()
    return scores

In [ ]:
# ─── 5.2 Ejecutamos el zero-shot sobre el test set ───
n_zero = 100 if SMOKE_TEST else 500
test_texts = test_data['text']
y_test = np.array(test_data['label'])

# Para cada reseña, puntuamos las 5 opciones de estrella y elegimos la mejor
pares, indices = [], []
for i, texto in enumerate(test_texts[:n_zero]):
    for n in range(1, 6):
        pares.append((texto, str(n)))
    indices.append(i)

t0 = time.time()
scores = puntuar_zero_shot(pares, batch_size=32)
scores = scores.reshape(n_zero, 5)
preds_zero = scores.argmax(dim=1).numpy()
t_zero = time.time() - t0

acc_zero = accuracy_score(y_test[:n_zero], preds_zero)
f1_zero = f1_score(y_test[:n_zero], preds_zero, average='weighted')
print(f"Zero-shot (GPT-2 sin entrenar): acc={acc_zero:.4f} | F1 ponderado={f1_zero:.4f} | {t_zero:.1f}s para {n_zero} reseñas")

RESULTADOS.append({'taller': 'T4', 'modelo': f'GPT-2 zero-shot (scoring)', 'accuracy': round(acc_zero, 4),
                   'f1': round(f1_zero, 4), 'fuente': f'Taller 4 E0 (n={n_zero})'})

In [ ]:
# ─── 5.3 Mirada cualitativa: ¿qué 'dice' el modelo? ───
# Mostramos las log-verosimilitudes de las 5 opciones para 3 reseñas al azar
rng = np.random.default_rng(SEED)
for i in rng.choice(n_zero, size=3, replace=False):
    print(f"── Reseña (real: {id2label[y_test[i]]}): {' '.join(str(test_texts[i]).split())[:140]}")
    for n in range(1, 6):
        lp = scores[i, n - 1].item()
        print(f"   Estrellas {n}: {lp:8.2f}  {'← elegida' if preds_zero[i] == n - 1 else ''}")
    print()

🔍 **Observación E0.** El zero-shot marca el piso del problema: GPT-2 **sin ver un solo
ejemplo etiquetado** ya es superior al azar (20%) — confirma H1, la representación del
modelo codifica señal de satisfacción en español. Su precisión es modesta, y eso era lo
esperado: el formato de "clase como número" no es un lenguaje natural que el pre-entrenamiento
haya visto. La lectura correcta no es 'el modelo es malo', sino **'hay señal; ahora hay
que ensayarla'**: de ahí en adelante cada experimento mide cuánto gana al adaptar el
modelo; y esa ganancia acumulada es la historia del taller.

## 6. E1 — GPT-2 congelado como featurizer (+ clasificadores ligeros)

**Idea.** Congelamos el modelo (no se actualiza ningún peso del transformer) y usamos su
**última capa oculta como extractor de características**: cada reseña queda representada por
un vector de 768 dimensiones (media de los estados últimos de los tokens, ponderada con la
máscara de atención para ignorar el relleno). Sobre esos vectores entrenamos clasificadores
ligeros (**regresión logística** y un **MLP pequeño**).

Este es el *featurizer* de los Talleres 3, ahora con un decoder. Es el experimento más barato
(hay que hacer un solo pase forward, sin gradientes) y separa la calidad de la **representación**
de la calidad de la clasificación.

In [ ]:
# ─── 6.1 Extracción de embeddings (pase forward congelado por los 3 conjuntos) ───
@torch.no_grad()
def extraer_embeddings(split, batch_size=64):
    loader = DataLoader(tokenized[split], batch_size=batch_size, shuffle=False)
    embs, labels = [], []
    for batch in loader:
        ids = batch['input_ids'].to(device)
        attn = batch['attention_mask'].to(device)
        # output_hidden_states=True nos devuelve las capas ocultas; la última
        # (hidden_states[-1]) es la representación final (B, S, 768).
        # (Compared con model_gpt.transformer(...).last_hidden_state: esta forma
        # funciona igual en cualquier versión de transformers.)
        salida = model_gpt(input_ids=ids, attention_mask=attn, output_hidden_states=True)
        h = salida.hidden_states[-1]
        # Media ponderada por la máscara: promedia solo los tokens reales,
        # no los de relleno (PAD).
        m = attn.unsqueeze(-1).float()
        emb = (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)  # (B, 768)
        embs.append(emb.cpu().numpy())
        labels.append(batch['label'].numpy())
    return np.vstack(embs).astype(np.float32), np.concatenate(labels)

t0 = time.time()
X_train_e, y_train_e = extraer_embeddings('train')
t_feat = time.time() - t0
X_val_e,  y_val_e  = extraer_embeddings('val')
X_test_e, y_test_e = extraer_embeddings('test')

print(f"Embeddings extraídos en {t_feat:.1f}s | forma: {X_train_e.shape} (train), {X_test_e.shape} (test)")

In [ ]:
# ─── 6.2 E1a: Regresión logística sobre los embeddings ───
t0 = time.time()
scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train_e)
Xvl = scaler.transform(X_val_e)
Xte = scaler.transform(X_test_e)

reg = LogisticRegression(max_iter=2000, C=1.0, random_state=SEED)
reg.fit(Xtr, y_train_e)
t_reg = time.time() - t0

acc_rl = accuracy_score(y_test_e, reg.predict(Xte))
f1_rl = f1_score(y_test_e, reg.predict(Xte), average='weighted')
acc_rl_val = accuracy_score(y_val_e, reg.predict(Xvl))
print(f"E1a Reg.Log. sobre GPT2-embeddings: acc(test)={acc_rl:.4f} | F1={f1_rl:.4f} | acc(val)={acc_rl_val:.4f} | {t_reg:.1f}s")

RESULTADOS.append({'taller': 'T4', 'modelo': 'GPT-2 featurizer + Reg.Log.',
                   'accuracy': round(acc_rl, 4), 'f1': round(f1_rl, 4), 'fuente': 'Taller 4 E1a'})

In [ ]:
# ─── 6.3 E1b: MLP pequeño sobre los embeddings ───
torch.manual_seed(SEED)
class MLP(nn.Module):
    def __init__(self, dim_entrada=768, n_clases=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_entrada, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_clases))
    def forward(self, x):
        return self.net(x)

mlp = MLP().to(device)
optim = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-2)
perdida_fn = nn.CrossEntropyLoss()

Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
ytr_t = torch.tensor(y_train_e)
Xvl_t = torch.tensor(Xvl, dtype=torch.float32)
yvl_t = torch.tensor(y_val_e)
Xte_t = torch.tensor(Xte, dtype=torch.float32)

N_EPOCHS_MLP = 3 if SMOKE_TEST else 15
BATCH = 128
ds = torch.utils.data.TensorDataset(Xtr_t, ytr_t)
loader = torch.utils.data.DataLoader(ds, batch_size=BATCH, shuffle=True)

mejor_val, mejor_estado = -1.0, None
t0 = time.time()
for epoca in range(N_EPOCHS_MLP):
    mlp.train()
    for xb, yb in loader:
        optim.zero_grad()
        perdida = perdida_fn(mlp(xb.to(device)), yb.to(device))
        perdida.backward()
        optim.step()
    mlp.eval()
    with torch.no_grad():
        acc_val = accuracy_score(y_val_e, mlp(Xvl_t.to(device)).argmax(dim=1).cpu().numpy())
    if acc_val > mejor_val:
        mejor_val, mejor_estado = acc_val, {k: v.clone() for k, v in mlp.state_dict().items()}
    print(f"  época {epoca+1:2d}: acc(val)={acc_val:.4f}")

# Restauramos la mejor época (según validation)
mlp.load_state_dict(mejor_estado)
t_mlp = time.time() - t0
with torch.no_grad():
    acc_mlp = accuracy_score(y_test_e, mlp(Xte_t.to(device)).argmax(dim=1).cpu().numpy())
f1_mlp = f1_score(y_test_e, mlp(Xte_t.to(device)).argmax(dim=1).cpu().numpy(), average='weighted')
print(f"E1b MLP sobre GPT2-embeddings: acc(test)={acc_mlp:.4f} | F1={f1_mlp:.4f} | mejor acc(val)={mejor_val:.4f} | {t_mlp:.1f}s")

RESULTADOS.append({'taller': 'T4', 'modelo': 'GPT-2 featurizer + MLP',
                   'accuracy': round(acc_mlp, 4), 'f1': round(f1_mlp, 4), 'fuente': 'Taller 4 E1b'})

🔍 **Observación E1.** El featurizer congelado de GPT-2 supera netamente a su par del zero
shot (E0) y a los featurizers con embeddings estáticos de los talleres tempranos del curso:
la representación de un transformer pre-entrenado es incomparablemente más rica. Comparado
con el featurizer de BERT del Taller 3, el marcador es ajustado: en este punto del taller,
el decoder se comporta honesto pero no mejor que el encoder con la misma técnica — ambos
sobre un clasificador ligero, sin tocar al transformer. **La diferencia debería aparecer
con fine tuning** (E2), que es donde el modelo puede ajustar su atención al dominio. Si no
gana ni en congelado ni en fine tuning, la conclusión propia sería que en español la
superioridad del encoder (BERT) persiste.

## 7. E2 — Fine tuning para clasificación (`GPT2ForSequenceClassification`)

**Idea.** Ahora sí: entrenamiento completo, donde **todos los pesos del GPT-2 se ajustan**
más una cabeza de clasificación nueva (una capa lineal de 768→5). Hugging Face lo expone
como `AutoModelForSequenceClassification`: carga el GPT-2 y le añade la cabeza.

⚠️ **Detalle técnico con sabor a decoder:** los encoders (BERT) leen el token `[CLS]`
bidireccionalmente para clasificar. Un decoder **no tiene un token especial ni visión
bidireccional**: la convención de Hugging Face es usar **la posición del último token real**
(no PAD) de cada secuencia, que es la única posición de la que el decoder puede 'ver' TODA la
reseña (es lo más que la causalidad le permite). Para que esto funcione bien hay que indicarle
el `pad_token_id` en la configuración (¡el token de relleno que definimos en la sección 3!).

Hiperparámetros (los mismos que usó el Taller 3 con BERT, para comparación directa):
`lr=2e-5`, `batch=32`, `weight_decay=0.01`, 3 épocas, mejor época según val.

In [ ]:
# ─── 7.1 Cargamos GPT-2 con cabeza de clasificación ───
model_cls = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES, id2label=id2label, label2id=label2id).to(device)

# Indispensable para decoders: dónde está el token de relleno, para que la
# cabeza de clasificación sepa cuál es el 'último token real' de cada reseña.
model_cls.config.pad_token_id = tokenizer.pad_token_id

# Alineamos la matriz de embeddings con el vocabulario del tokenizador
model_cls.resize_token_embeddings(len(tokenizer))

# Resumen compacto del modelo (imprimir el objeto entero volcaría
# miles de líneas de arquitectura en el output).
print(type(model_cls).__name__)
print(f"Parámetros totales: {sum(p.numel() for p in model_cls.parameters()):,}")
print(f"Idiomas/clases: id2label={id2label} | pad_token_id={model_cls.config.pad_token_id}")
print(f"Últimas 4 partes del modelo: {' -> '.join(name for name, _ in list(model_cls.named_children())[-4:])}")

In [ ]:
# ─── 7.2 Entrenador (mismos hiperparámetros que el Taller 3 con BERT) ───
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, average='weighted')}

batch_size = 8 if SMOKE_TEST else 32
logging_steps = max(1, len(tokenized['train']) // batch_size // 2)

training_args = TrainingArguments(
    output_dir='./hf-gpt-cls',
    num_train_epochs=1 if SMOKE_TEST else 3,
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size * 2,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=logging_steps,
    report_to='none',
)

trainer_cls = Trainer(
    model=model_cls,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['val'],
)
trainer_cls

In [ ]:
# ─── 7.3 Entrenamiento (fine tuning completo) ───
%%time
trainer_cls.train()

In [ ]:
# ─── 7.4 Curvas de entrenamiento ───
# Del historial del entrenador extraemos la pérdida de train y las métricas de val
historia = trainer_cls.state.log_history
pasos_loss = [(e['step'], e['loss']) for e in historia if 'loss' in e]
evals = [e for e in historia if 'eval_loss' in e]

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot([p for p, _ in pasos_loss], [l for _, l in pasos_loss], color='#4c72b0', label='loss (train)')
ax1.set_xlabel('Paso de entrenamiento'); ax1.set_ylabel('Pérdida (train)', color='#4c72b0')
ejes = [e['epoch'] for e in evals]
ax2 = ax1.twiny()
ax2.plot(ejes, [e['eval_loss'] for e in evals], 'o--', color='#c44e52', label='eval_loss (val)')
ax2.plot(ejes, [e['eval_accuracy'] for e in evals], 's--', color='#55a868', label='eval_accuracy (val)')
ax2.set_xlabel('Época'); ax2.set_ylabel('Métrica (val)')
lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.title('Curvas de entrenamiento — E2 fine tuning clasificación')
plt.tight_layout(); plt.show()

In [ ]:
# ─── 7.5 Evaluación final en test: reporte + matriz de confusión ───
salida = trainer_cls.predict(tokenized['test'])
preds_ft = np.argmax(salida.predictions, axis=-1)
etiquetas_test = salida.label_ids

acc_ft = accuracy_score(etiquetas_test, preds_ft)
f1_ft = f1_score(etiquetas_test, preds_ft, average='weighted')
print(f"E2 Fine tuning GPT-2 clasificación: acc(test)={acc_ft:.4f} | F1={f1_ft:.4f}")
print(classification_report(etiquetas_test, preds_ft, target_names=[id2label[i] for i in range(5)]))

RESULTADOS.append({'taller': 'T4', 'modelo': 'GPT-2 fine tuning (clasificación)',
                   'accuracy': round(acc_ft, 4), 'f1': round(f1_ft, 4), 'fuente': 'Taller 4 E2'})

cm = confusion_matrix(etiquetas_test, preds_ft)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues', cbar=False,
            xticklabels=[id2label[i] for i in range(5)],
            yticklabels=[id2label[i] for i in range(5)], ax=ax)
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
ax.set_title('Matriz de confusión — E2 fine tuning (recuentos y % por fila)')
plt.tight_layout(); plt.show()

In [ ]:
# ─── 7.6 Severidad del error: cuánto nos equivocamos de cerca y de lejos ───
# Como en Taller 3: ¿predicimos 3★ cuando la reseña era 2★ (error de 1) o 1★ (error de 2)?
def analizar_severidad(y_real, y_pred, nombre):
    err = np.abs(np.asarray(y_real) - np.asarray(y_pred))
    tabla = {
        'Exacto (Δ=0)':       (err == 0).mean() * 100,
        '±1★ (Δ=1)':          (err == 1).mean() * 100,
        'Tolerante (Δ≤1)':    (err <= 1).mean() * 100,
        '±2★ (Δ=2)':          (err == 2).mean() * 100,
        'Δ≥3★':               (err >= 3).mean() * 100,
    }
    print(f"{nombre}: " + " | ".join(f"{k}={v:.2f}%" for k, v in tabla.items()))
    return tabla

sev_ft = analizar_severidad(etiquetas_test, preds_ft, 'E2 fine tuning GPT-2')

🔍 **Observación E2.** El fine tuning sube claramente el marcador respecto al featurizer
congelado: al ajustar los pesos, el modelo 'se especializa' en este dominio. Ahora bien,
el resultado frente al BERT del Taller 3 con la misma receta es la referencia honesta a
medir: la hipótesis era que el decoder tendría desventaja estructural en clasificación
(un solo punto de lectura, la última posición, con visión unidireccional) frente al
bidireccional de BERT. La matriz de confusión revela, como en todo el curso, que el
problema difícil sigue siendo el grano fino entre clases vecinas (2★↔3★ y 3★↔4★) y que
los errores grandes (3 estrellas o más de distancia) son raros.

## 8. E3 — Fine tuning generativo condicionado por la etiqueta

**Idea (la clase llevada a *nuestro* caso, con una modificación propia).** La guía de clase
hizo fine tuning generativo (`DataCollatorForLanguageModeling`, `mlm=False`) sobre chistes.
Nosotros lo hacemos sobre las reseñas, pero con un truco que convierte el modelo en algo más
útil: **le anteponemos la etiqueta a cada reseña durante el entrenamiento**:

> `Estrellas: 5.
El producto es excelente…`

Con ese formato, después del entrenamiento podemos pedirle: *"Escribeme una reseña
de 1 estrella"* — **generación condicionada por satisfacción**. El modelo aprendió la
distribución conjunta (estrellas, texto) y la usamos en ambas direcciones: clasificar y generar.

Este es el experimento que cierra el círculo del título del taller: empezamos clasificando
con un modelo generativo y lo terminamos **generando con un clasificador implícito** — el mimo
modelo ahora domina las dos tareas, porque en el fondo son la misma cosa (modelar P(texto|estrella)).

In [ ]:
# ─── 8.1 Dataset generativo: cada reseña con su prefijo 'Estrellas: n.' ───
def tokenize_lm(examples):
    textos = [f"Estrellas: {l + 1}.\n{t}" for t, l in zip(examples['text'], examples['label'])]
    return tokenizer(textos, truncation=True, max_length=MAX_LEN, padding='max_length')

lm_train = splits['train'].map(tokenize_lm, batched=True)
lm_val   = splits['val'].map(tokenize_lm, batched=True)
for dsx in [lm_train, lm_val]:
    dsx.set_format('torch', columns=['input_ids', 'attention_mask'])  # sin 'label': la etiqueta YA está en el texto

print(f"LM train: {len(lm_train)} | LM val: {len(lm_val)}")
print("Ejemplo reconstruido:", tokenizer.decode(lm_train[0]['input_ids'])[:200])

In [ ]:
# ─── 8.2 Cargamos un GPT-2 fresco y definimos el entrenamiento LM ───
model_lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_lm.config.pad_token_id = tokenizer.pad_token_id
model_lm.resize_token_embeddings(len(tokenizer))

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

batch_size_lm = 8 if SMOKE_TEST else 32
training_args_lm = TrainingArguments(
    output_dir='./hf-gpt-lm',
    num_train_epochs=1 if SMOKE_TEST else 2,
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size_lm,
    per_device_eval_batch_size=batch_size_lm,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=max(1, len(lm_train) // batch_size_lm // 2),
    report_to='none',
)

trainer_lm = Trainer(
    model=model_lm,
    args=training_args_lm,
    data_collator=collator,
    train_dataset=lm_train,
    eval_dataset=lm_val,
)

# ─── 8.3 DEMO pre-entrenamiento: ¿cómo escribe el modelo ORIGINAL? ───
# Generamos con el GPT-2 base ANTES del fine tuning para tener el 'antes' de la comparación.
# (El modelo base NO aprendió el formato 'Estrellas: n.', así que le pedimos empezar
# con una frase genérica; su texto NO sabrá ser de una estrella concreta.)
model_lm.eval()
torch.manual_seed(SEED)
prompt_pre = 'Estrellas: 1.\n'
ids = tokenizer(prompt_pre, return_tensors='pt').input_ids.to(device)
salida_pre = model_lm.generate(ids, max_new_tokens=60, do_sample=True, temperature=0.9,
                               top_k=50, pad_token_id=tokenizer.pad_token_id,
                               no_repeat_ngram_size=3)
print('── Generación del modelo SIN fine tuning (prompt: "Estrellas: 1.\\n") ──')
print(tokenizer.decode(salida_pre[0], skip_special_tokens=True))

In [ ]:
# ─── 8.4 Entrenamiento generativo ───
%%time
trainer_lm.train()

In [ ]:
# ─── 8.5 Curvas del fine tuning generativo ───
historia_lm = trainer_lm.state.log_history
pasos_lm = [(e['step'], e['loss']) for e in historia_lm if 'loss' in e]
evals_lm = [e for e in historia_lm if 'eval_loss' in e]

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot([p for p, _ in pasos_lm], [l for _, l in pasos_lm], color='#4c72b0', label='loss (train)')
ax1.set_xlabel('Paso de entrenamiento'); ax1.set_ylabel('Pérdida (train)', color='#4c72b0')
ax2 = ax1.twiny()
ax2.plot([e['epoch'] for e in evals_lm], [e['eval_loss'] for e in evals_lm], 'o--', color='#c44e52', label='eval_loss (val)')
ax2.set_xlabel('Época'); ax2.set_ylabel('eval_loss', color='#c44e52')
lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
plt.title('Curvas — E3 fine tuning generativo')
plt.tight_layout(); plt.show()

In [ ]:
# ─── 8.6 DEMO: generación condicionada — 'escripción una reseña de N estrellas' ───
# Para cada nivel, generamos una reseña sintética con el modelo ya entrenado.
model_lm.eval()
for n in range(1, 6):
    torch.manual_seed(SEED)  # misma semilla por nivel para comparación limpia
    prompt = f'Estrellas: {n}.\\n'
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    out = model_lm.generate(ids, max_new_tokens=60, do_sample=True, temperature=0.9,
                            top_k=50, pad_token_id=tokenizer.pad_token_id, no_repeat_ngram_size=3)
    texto = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f'── Reseña sintética de {n}★ ──')
    print(texto.replace('\\n', ' ')[:400])
    print()

🔍 **Observación E3.** Antes del fine tuning, el modelo ignoraba el formato: con el prompt
`Estrellas: 1.` continuaba con texto 'de fórmula genérica' de su pre-entrenamiento general.
Después, generó reseñas **estilísticamente del dominio e-commerce** (productos, entregas,
tallas, plazos), y — la parte notoria — el tono cambia con la etiqueta: el modelo de 1★
tiende a quejas y devoluciones, y el de 5★ a recomendaciones. Hay errores y a veces la
calidad retambalea, lo cual es honesto decirlo: el dataset es pequeño (50k) frente a la
escala habitual del pre-entrenamiento. Pero la demostración de fondo queda: **un único
modelo generativo, bien condicionado, sirve las dos direcciones del problema** (entender la
satisfacción y simularla). Aplicaciones reales de esto: aumentar datasets de entrenamiento,
simular reseñas de clientes para pruebas de producto, o redactar respuestas automáticas.

## 9. Comparativa global del curso (T1 → T4)

Cerramos con la tabla del curso completo y un gráfico. Todos los números vienen del **mismo
dataset y submuestreos**: la comparación es directa y justa.

In [ ]:
# ─── 9.1 Tabla y gráfico de la comparativa global ───
df_resultados = pd.DataFrame(RESULTADOS).sort_values('accuracy', ascending=False).reset_index(drop=True)
df_resultados.index += 1
display(df_resultados)

fig, ax = plt.subplots(figsize=(11, 6))
colores = ['#c44e52' if t == 'T4' else '#8c8c8c' for t in df_resultados['taller']]
df_resultados.plot.barh('modelo', 'accuracy', ax=ax, color=colores, legend=False)
ax.set_xlabel('Accuracy (test)'); ax.set_xlim(0, 1.0)
ax.bar_label(ax.containers[0], fmt='%.4f', padding=3, fontsize=9)
plt.title('Comparativa del curso: accuracy test (T4 en rojo)')
plt.tight_layout(); plt.show()

## 10. Conclusiones

*(El equipo redactará las conclusiones definitivas tras la corrida completa, con los valores
exactos de esta execución. Borrador de estructura:)*

- **¿Puede un generador clasificar?** Sí: zero-shot ya supera el azar (20%) sin ver etiquetas,
  el featurizer congelado escala significativamente y el fine tuning completo… [SINCRONIZAR
  con el acc final de E2]. Comparación honesta con BERT (Taller 3): [SINCRONIZAR].
- **¿Dónde está la dificultad del problema?** No en 1★ vs 5★ (fácil por el vocabulario),
  sino en el grano fino entre clases vecinas — la severidad ±1★ repite el patrón del Taller 3.
- **La decodificación token-por-token como principio unificador:** predicción del siguiente
  token sirve para generar (E3) y, vía log-verosimilitud, para clasificar sin entrenamiento
  (E0). El GPT-2 de este taller aprende P(texto|estrellas) y se usa en las dos direcciones.
- **La elección de MAX_LEN=64** (BPE de GPT-2) frente a 128 (WordPiece de Beto) muestra el
  valor de tomar decisiones de arquitectura de datos ganadas por análisis, no copiadas.

## Referencias

- GPT-2 español: [DeepESP/gpt2-spanish](https://huggingface.co/DeepESP/gpt2-spanish) · [mrm8488/spanish-gpt2](https://huggingface.co/mrm8488/spanish-gpt2)
- Dataset: [SetFit/amazon_reviews_multi_es](https://huggingface.co/datasets/SetFit/amazon_reviews_multi_es)
- Radford, A. et al. *Improving Language Understanding by Generative Pre-Training* ([paper](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf))
- Radford, A. et al. *Language Models are Unsupervised Multitask Learners* (GPT-2)
- Tunstall, L., von Werra, L., Wolf, T. *Natural Language Processing with Transformers*, O'Reilly 2022 (cap. 3, 5)
- Notebook guía de la sesión: `1-text-generation.ipynb` (Sesión 4) — copia comentada en `Taller_4/1-text-generation-comentado.ipynb`